<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 22 · VaR, Expected Shortfall, and Stress Testing

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch22_var_expected_shortfall_stress_testing.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Chapter 21's deterministic shocks become an empirical loss distribution here. You will map a frozen panel of 40 daily rate and spread changes through a fixed exposure vector to produce positive one-day loss amounts, then compute historical VaR and expected shortfall at the 95 percent level, a covariance-based parametric VaR, and losses under four named stress scenarios. You will also trace an expanding-window backtest in which every forecast uses only strictly earlier observations, and practice keeping statistical tail measures and named scenarios conceptually separate when reporting.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script converts 40 daily factor changes into one-day loss amounts with the frozen DV01 and CS01 exposure vector, then prints historical VaR and ES at 95 percent, a normal parametric VaR, and four named stress losses. Historical VaR is 120,890 and ES is 155,600, while the liquidity-crisis stress reaches 289,150, above every statistical measure in the report.

In [ ]:
SCRIPT = ROOT / "code" / "ch22_var_expected_shortfall_stress_testing.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Match each printed metric to its convention before quoting it.

- Losses are positive one-day currency amounts built from exposures of 870, 1,470, and 1,540 per basis point plus a CS01 of 1,130.
- Historical VaR 95 is 120,890, the lower 95 percent quantile of the 40 observations.
- ES 95 is 155,600, the mean of the worst two losses, matching the ceiling of 5 percent times 40 tail convention.
- Parametric VaR 95 is 108,204.71 under a normal assumption with the sample mean loss; it sits below the historical measures on this sample.
- The backtest expands its window row by row, so each VaR uses only strictly earlier losses; the first exception appears on 2025-02-07.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch22_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Count exceedances of a VaR estimated on these same observations. This is an in-sample diagnostic, not an out-of-sample backtest; 5% is only a nominal reference for the small sample.

In [ ]:
import numpy as np

changes = pd.read_csv(DATA / "ch22_factor_changes.csv")
exposures = pd.read_csv(DATA / "ch22_portfolio_exposures.csv")
exposures = exposures.set_index("factor").exposure
factors = ["rate_2y_bp", "rate_5y_bp", "rate_10y_bp", "spread_bp"]
losses = (changes[factors] * exposures.reindex(factors)).sum(axis=1)
var_95 = np.quantile(losses, 0.95, method="lower")
exceptions = losses.gt(var_95)
pd.DataFrame({
    "var_95": [var_95],
    "observations": [len(losses)],
    "exceptions": [int(exceptions.sum())],
    "expected_exceptions": [0.05 * len(losses)],
    "exception_rate": [exceptions.mean()],
}).round(4)

## Mini-lab: Historical VaR and expected shortfall at 97.5 percent

A seeded sample of 200 one-day profit-and-loss observations stands in for a fixed-income book; positive values are losses, negative values are net gains, and the standard deviation of 120,000 currency units is illustrative. The 97.5 percent historical VaR is the lower quantile of the empirical loss distribution. Expected shortfall averages the worst tail observations, where the tail count is the ceiling of 2.5 percent times 200, or five losses, computed with the chapter's tie-safe epsilon guard. The horizon is one day.

In [ ]:
import numpy as np
from math import ceil

rng = np.random.default_rng(42)  # fixed seed, repeatable draws
losses = rng.normal(0.0, 120000.0, 200)  # one-day profit and loss
level = 0.975  # confidence level
var = np.quantile(losses, level, method="lower")  # tail threshold
count = ceil((1.0 - level) * len(losses) - 1e-12)  # tie-safe count
es = float(np.sort(losses)[-count:].mean())  # mean of worst tail
print(f"observations: {len(losses)}  tail count: {count}")
print(f"97.5% VaR: {var:,.2f}")
print(f"97.5% ES: {es:,.2f}")
print(f"worst loss: {losses.max():,.2f}")

### What to observe

With seed 42, the 97.5 percent VaR is 206,839.89 and expected shortfall is 262,767.37, the mean of the worst 5 of 200 losses; the worst observation is 349,663.50. ES sits above VaR because it averages the losses beyond the threshold instead of merely locating it.

## Exercises

Work these from the script output and the frozen factor panel; state the loss sign, level, and horizon with every result.

1. **Rebuild the loss series.** Recompute the 40 daily losses from the factor changes and the frozen exposures, and confirm positive values are losses. *Hint:* Multiply each factor column by its per-basis-point exposure and sum across columns.
2. **Verify the tail convention.** Check that ES 95 of 155,600 is exactly the mean of the worst two losses. *Hint:* Use the ceiling of 5 percent of 40 observations for the tail count.
3. **Challenge the parametric number.** Recompute parametric VaR from the factor covariance matrix and the sample mean loss, and explain why it can fall below the historical VaR. *Hint:* The normal model assumes thin-tailed, stable factor changes.
4. **Read the backtest.** Count exceptions across the expanding-window rows and explain why 40 observations make the count weak evidence. *Hint:* Each forecast starts from at least 20 training observations.
5. **Add a stress.** Design a scenario combining a 55 bp long-end sell-off with 120 bp of spread widening, and compute its loss. *Hint:* Apply the same exposure vector the script uses.

For more detail, see Chapter 22 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">